# Bank Customer Churn Dataset - Data Preprocessing

This notebook performs a standard preprocessing workflow for the **Bank Customer Churn** dataset.

## Goals
- inspect the dataset
- check data quality
- remove non-predictive identifier columns
- encode categorical variables
- split the data into train and test sets
- scale numeric features for models such as **Logistic Regression** and **SVM**
- save the processed files for model training

> Target column: **Exited**


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load the dataset

In [ ]:
# Update this path only if your file name changes
file_path = Path("639a58d4-bb1f-4832-a062-70f4d8a3903f.csv")

df = pd.read_csv(file_path)
df.head()


## 2. Basic inspection

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Understand the target variable

In [ ]:
print(df['Exited'].value_counts())
print("\nClass proportions:")
print(df['Exited'].value_counts(normalize=True).round(4))


## 4. Drop identifier / non-predictive columns

These columns are removed because they do not meaningfully help prediction:

- **RowNumber**: just row index information
- **CustomerId**: unique customer identifier
- **Surname**: customer name-like text with very high cardinality

Keeping these can add noise and reduce model quality.


In [ ]:
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df_clean = df.drop(columns=columns_to_drop)

df_clean.head()


## 5. Separate features and target

In [ ]:
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## 6. Identify categorical and numerical columns

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


## 7. Train-test split

We use **stratify=y** to preserve the churn / non-churn class ratio in both train and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 8. Build preprocessing pipelines

### Numerical columns
- impute missing values with median
- scale using `StandardScaler`

### Categorical columns
- impute missing values with most frequent value
- encode with `OneHotEncoder`

This is the **standard and safer** preprocessing approach for this dataset.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

preprocessor


## 9. Fit preprocessing on training data only

This avoids **data leakage**.


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)


## 10. Convert processed arrays back to DataFrames

In [ ]:
# Get final feature names after one-hot encoding
cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=all_feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=all_feature_names,
    index=X_test.index
)

X_train_processed_df.head()


## 11. Final processed training and testing datasets

In [ ]:
train_processed = X_train_processed_df.copy()
train_processed['Exited'] = y_train.values

test_processed = X_test_processed_df.copy()
test_processed['Exited'] = y_test.values

print("Processed training set:")
display(train_processed.head())

print("Processed test set:")
display(test_processed.head())


## 12. Save processed files

In [ ]:
output_dir = Path("processed_output")
output_dir.mkdir(exist_ok=True)

train_processed.to_csv(output_dir / "train_processed.csv", index=False)
test_processed.to_csv(output_dir / "test_processed.csv", index=False)
X_train_processed_df.to_csv(output_dir / "X_train_processed.csv", index=False)
X_test_processed_df.to_csv(output_dir / "X_test_processed.csv", index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("Files saved in:", output_dir.resolve())
print(sorted([p.name for p in output_dir.iterdir()]))


## 13. Notes

### Why this preprocessing is correct for your dataset
- there are **no missing values**, but imputation steps are kept for robustness
- there are **no duplicate rows**
- `RowNumber`, `CustomerId`, and `Surname` are dropped because they are not strong predictive features in a standard churn model
- `Geography` and `Gender` are categorical, so **One-Hot Encoding** is preferred
- scaling is useful for **Logistic Regression** and **SVM**
- this processed data can be directly used for:
  - Logistic Regression
  - SVM
  - Decision Tree
  - Random Forest

### Important note
For **Decision Tree** and **Random Forest**, scaling is not strictly required, but keeping one common processed dataset is convenient for comparison across models.
